In [1]:
from pathlib import Path
from scipy import sparse

import pandas as pd
import numpy as np
import xarray as xr
import glob
import os


In [ ]:
###############################################################################################################################################
# 1980
###############################################################################################################################################

In [ ]:
# u10_dir = Path("/lfs/archive/Reanalysis/ERA5/SFC/1hr/u10/1980")
# v10_dir = Path("/lfs/archive/Reanalysis/ERA5/SFC/1hr/v10/1980")

# out_dir = Path("/lfs/home/yanlan/climada/tc-risk/ERA5/sqrt_u10_2_v10_2")

# for month in range(1, 13):

#     yyyymm = f"1980{month:02d}"

#     u10_file = u10_dir / f"ERA5_SFC_u10_{yyyymm}_r1440x721_1hr.nc"
#     v10_file = v10_dir / f"ERA5_SFC_v10_{yyyymm}_r1440x721_1hr.nc"

#     out_file = out_dir / f"ERA5_SFC_sqrt_{yyyymm}_r1440x721_1hr.nc"


#     u10_ds = xr.open_dataset(u10_file)
#     v10_ds = xr.open_dataset(v10_file)

#     # assert u10_ds["time"].equals(v10_ds["time"])
#     # assert u10_ds["latitude"].equals(v10_ds["latitude"])
#     # assert u10_ds["longitude"].equals(v10_ds["longitude"])
    
#     sqrt_wind = ((u10_ds["u10"]**2 + v10_ds["v10"]**2)**0.5).rename("sqrt_wind_speed")
#     sqrt_wind.attrs["long_name"] = "sqrt(u10^2 + v10^2)"
#     sqrt_wind.attrs["units"] = "m s**-1"
    
#     sqrt_wind.to_netcdf(out_file)


#     u10_ds.close()
#     v10_ds.close()

In [ ]:
###############################################################################################################################################
# 1981~
###############################################################################################################################################

In [ ]:
# years = range(1981, 1991)
# out_dir = Path("/lfs/home/yanlan/climada/tc-risk/ERA5/sqrt_u10_2_v10_2")


# for y in years:
#     u10_dir = Path(f"/lfs/archive/Reanalysis/ERA5/SFC/1hr/u10/{y}")
#     v10_dir = Path(f"/lfs/archive/Reanalysis/ERA5/SFC/1hr/v10/{y}")
#     for month in range(1, 13):
    
#         yyyymm = f"{y}{month:02d}"
    
#         u10_file = u10_dir / f"ERA5_SFC_u10_{yyyymm}_r1440x721_1hr.nc"
#         v10_file = v10_dir / f"ERA5_SFC_v10_{yyyymm}_r1440x721_1hr.nc"
    
#         out_file = out_dir / f"ERA5_SFC_sqrt_{yyyymm}_r1440x721_1hr.nc"
    
#         with xr.open_dataset(u10_file) as u10_ds, xr.open_dataset(v10_file) as v10_ds:
            
#             # assert u10_ds["time"].equals(v10_ds["time"])
#             # assert u10_ds["latitude"].equals(v10_ds["latitude"])
#             # assert u10_ds["longitude"].equals(v10_ds["longitude"])
            
#             sqrt_wind = ((u10_ds["u10"]**2 + v10_ds["v10"]**2)**0.5)
#             sqrt_wind = sqrt_wind.astype("float32")
#             sqrt_wind = sqrt_wind.rename("sqrt_wind_speed")
            
#             sqrt_wind.attrs["long_name"] = "sqrt(u10^2 + v10^2)"
#             sqrt_wind.attrs["units"] = "m s**-1"
            
#             sqrt_wind.to_netcdf(out_file)
        


In [2]:
###############################################################################################################################################
# subsetting (TAIWAN), climatology and anomaly
###############################################################################################################################################

In [ ]:
###############################################################################################################################################
# go to terminal -> jupyter labextension disable @lckr/jupyterlab_variableinspector --level=user
###############################################################################################################################################

In [33]:
lat_min, lat_max = 20.0, 29.0 
lon_min, lon_max = 117.0, 126.0 
def read_one(f):
    with xr.open_dataset(f) as d:
        return d["sqrt_wind_speed"].sel(latitude=slice(lat_max, lat_min), # ERA5 latitude is descending: 90 -> -90
                                        longitude=slice(lon_min, lon_max)
                                       ).load()

In [34]:
from concurrent.futures import ProcessPoolExecutor

all_files = sorted(
    Path("/lfs/home/yanlan/climada/tc-risk/ERA5/sqrt_u10_2_v10_2").glob("ERA5_SFC_sqrt_*.nc") )


with ProcessPoolExecutor(max_workers=16) as pool:
    pieces = list(pool.map(read_one, all_files)) # pool.map(read_one, all_files) -> a generator object
                                                 # pieces is a list of xarray.DataArrays by month over 1979~2020

wind_tw = xr.concat(pieces, dim="time").sortby("time")

# hourly climatology over 1979-2020
hourly_clim_tw = wind_tw.groupby("time.hour").mean("time")
hourly_clim_tw = hourly_clim_tw.astype("float32")
hourly_clim_tw = hourly_clim_tw.rename("sqrt_wind_speed_hourly_climatology")



# anomaly
wind_anom_tw = wind_tw.groupby("time.hour") - hourly_clim_tw # ! (24 groups, 15341 days ,37, 37) minus (24 groups, 37, 37)
                                                             # (29.0-20.0)/0.25 + 1 = 37
                                                             # (126.0-117.0)/0.25 + 1 = 37

wind_anom_tw = wind_anom_tw.astype("float32")
wind_anom_tw = wind_anom_tw.rename("sqrt_wind_speed_anomaly")
wind_anom_tw = wind_anom_tw.drop_vars("hour")




In [35]:
wind_tw

<xarray.DataArray 'sqrt_wind_speed' (time: 368184, latitude: 37, longitude: 37)> Size: 2GB
array([[[ 1.7834533 ,  1.5952629 ,  1.1856768 , ...,  9.661866  ,
          9.687716  ,  9.462225  ],
        [ 1.7644638 ,  1.6142517 ,  1.2977439 , ...,  9.857502  ,
          9.568268  ,  9.228493  ],
        [ 1.7637814 ,  1.6933113 ,  1.5102644 , ...,  9.579314  ,
          9.264062  ,  9.079899  ],
        ...,
        [11.040437  , 11.115386  , 11.150941  , ...,  7.1661224 ,
          7.2050962 ,  7.234408  ],
        [11.237288  , 11.290733  , 11.355299  , ...,  7.5159106 ,
          7.5186877 ,  7.505623  ],
        [11.374761  , 11.424621  , 11.474971  , ...,  7.822444  ,
          7.7987375 ,  7.8255596 ]],

       [[ 1.9469618 ,  1.7458495 ,  1.3596693 , ...,  9.019624  ,
          9.174955  ,  9.132617  ],
        [ 1.9518667 ,  1.794914  ,  1.4836265 , ...,  9.41382   ,
          9.253417  ,  9.109319  ],
        [ 1.9202529 ,  1.8124406 ,  1.6300044 , ...,  9.565928  ,
          9.256021  ,  8.910336  ],
...
        [11.962005  , 12.060895  , 12.209813  , ...,  9.377671  ,
          9.448829  ,  9.455006  ],
        [12.53341   , 12.627435  , 12.761123  , ...,  9.743674  ,
          9.829945  ,  9.771199  ],
        [13.169107  , 13.20188   , 13.104495  , ..., 10.264536  ,
         10.264494  , 10.205063  ]],

       [[ 1.442939  ,  1.1687579 ,  1.1332628 , ...,  7.4811535 ,
          7.4257255 ,  7.4640465 ],
        [ 1.3703108 ,  1.1663879 ,  1.291633  , ...,  7.4674454 ,
          7.577355  ,  7.675488  ],
        [ 1.2353833 ,  0.84384704,  0.585249  , ...,  7.5679717 ,
          7.728477  ,  7.968048  ],
        ...,
        [12.320544  , 12.354222  , 12.369209  , ...,  9.615732  ,
          9.612619  ,  9.631957  ],
        [12.79274   , 12.8276    , 12.86253   , ...,  9.906588  ,
          9.88314   ,  9.836875  ],
        [13.257011  , 13.270629  , 13.275356  , ..., 10.398113  ,
         10.262864  , 10.18349   ]]], dtype=float32)
Coordinates:
  * longitude  (longitude) float32 148B 117.0 117.2 117.5 ... 125.5 125.8 126.0
  * latitude   (latitude) float32 148B 29.0 28.75 28.5 28.25 ... 20.5 20.25 20.0
  * time       (time) datetime64[ns] 3MB 1979-01-01 ... 2020-12-31T23:00:00
Attributes:
    long_name:  sqrt(u10^2 + v10^2)
    units:      m s**-1

In [36]:
wind_tw.groupby("time.hour")

<DataArrayGroupBy, grouped over 1 grouper(s), 24 groups in total:
    'hour': UniqueGrouper('hour'), 24/24 groups with labels 0, 1, 2, 3, 4, 5, ..., 19, 20, 21, 22, 23>

In [37]:
hourly_clim_tw

<xarray.DataArray 'sqrt_wind_speed_hourly_climatology' (hour: 24, latitude: 37,
                                                        longitude: 37)> Size: 131kB
array([[[2.3947895, 2.0840409, 1.8670886, ..., 7.0558243, 7.059224 ,
         7.059332 ],
        [2.2515666, 1.9654322, 1.8340514, ..., 7.0917263, 7.0905323,
         7.101091 ],
        [2.151277 , 1.8770136, 1.7689316, ..., 7.1164765, 7.1182575,
         7.124392 ],
        ...,
        [7.6000924, 7.610016 , 7.6197205, ..., 7.184582 , 7.180312 ,
         7.1601815],
        [7.613598 , 7.6243367, 7.649394 , ..., 7.1670523, 7.163242 ,
         7.1358776],
        [7.6259494, 7.641706 , 7.672934 , ..., 7.143592 , 7.138076 ,
         7.1314836]],

       [[2.4857752, 2.2140508, 2.0332932, ..., 7.0559883, 7.0580072,
         7.0569973],
        [2.3570824, 2.1111596, 1.9712261, ..., 7.094803 , 7.0923433,
         7.101748 ],
        [2.2667959, 2.0437098, 1.9344013, ..., 7.122687 , 7.125007 ,
         7.130047 ],
...
        [7.4278584, 7.4375625, 7.442681 , ..., 6.9898076, 6.9867144,
         6.972324 ],
        [7.446913 , 7.4534454, 7.473399 , ..., 6.969619 , 6.966077 ,
         6.942308 ],
        [7.457559 , 7.4699917, 7.499151 , ..., 6.943901 , 6.9420285,
         6.9386215]],

       [[2.325106 , 2.018869 , 1.7946959, ..., 7.039038 , 7.042391 ,
         7.043498 ],
        [2.1890035, 1.9022875, 1.7485999, ..., 7.068705 , 7.0690207,
         7.0820336],
        [2.0978796, 1.7987428, 1.674053 , ..., 7.09191  , 7.09668  ,
         7.105876 ],
        ...,
        [7.52503  , 7.537219 , 7.5438848, ..., 7.105534 , 7.102801 ,
         7.0830035],
        [7.540537 , 7.5487175, 7.5703363, ..., 7.0865803, 7.0831075,
         7.0555024],
        [7.550331 , 7.5637083, 7.592772 , ..., 7.0611305, 7.058056 ,
         7.0527267]]], dtype=float32)
Coordinates:
  * longitude  (longitude) float32 148B 117.0 117.2 117.5 ... 125.5 125.8 126.0
  * latitude   (latitude) float32 148B 29.0 28.75 28.5 28.25 ... 20.5 20.25 20.0
  * hour       (hour) int64 192B 0 1 2 3 4 5 6 7 8 ... 16 17 18 19 20 21 22 23
Attributes:
    long_name:  sqrt(u10^2 + v10^2)
    units:      m s**-1

In [38]:
wind_anom_tw

<xarray.DataArray 'sqrt_wind_speed_anomaly' (time: 368184, latitude: 37,
                                             longitude: 37)> Size: 2GB
array([[[-0.6113361 , -0.488778  , -0.68141174, ...,  2.606042  ,
          2.6284914 ,  2.402893  ],
        [-0.48710287, -0.35118043, -0.53630745, ...,  2.7657757 ,
          2.4777355 ,  2.1274018 ],
        [-0.38749564, -0.18370223, -0.25866723, ...,  2.4628377 ,
          2.1458044 ,  1.9555068 ],
        ...,
        [ 3.4403443 ,  3.5053701 ,  3.5312204 , ..., -0.0184598 ,
          0.02478409,  0.07422638],
        [ 3.6236906 ,  3.6663966 ,  3.705905  , ...,  0.34885836,
          0.35544586,  0.36974525],
        [ 3.7488112 ,  3.7829146 ,  3.8020368 , ...,  0.6788521 ,
          0.6606617 ,  0.69407606]],

       [[-0.5388135 , -0.46820128, -0.6736239 , ...,  1.9636354 ,
          2.1169481 ,  2.0756197 ],
        [-0.40521562, -0.31624556, -0.4875996 , ...,  2.3190174 ,
          2.1610737 ,  2.0075707 ],
        [-0.34654295, -0.23126912, -0.30439687, ...,  2.4432416 ,
          2.1310134 ,  1.7802887 ],
...
        [ 4.5341463 ,  4.6233325 ,  4.7671323 , ...,  2.3878636 ,
          2.4621143 ,  2.4826818 ],
        [ 5.0864973 ,  5.1739893 ,  5.2877235 , ...,  2.7740555 ,
          2.8638678 ,  2.8288913 ],
        [ 5.7115483 ,  5.731889  ,  5.605344  , ...,  3.3206348 ,
          3.3224654 ,  3.2664413 ]],

       [[-0.88216686, -0.850111  , -0.6614331 , ...,  0.4421153 ,
          0.38333464,  0.42054844],
        [-0.8186927 , -0.73589957, -0.45696688, ...,  0.3987403 ,
          0.50833416,  0.59345436],
        [-0.8624964 , -0.95489573, -1.088804  , ...,  0.47606182,
          0.63179684,  0.8621721 ],
        ...,
        [ 4.795514  ,  4.8170033 ,  4.8253245 , ...,  2.510198  ,
          2.5098186 ,  2.5489535 ],
        [ 5.252203  ,  5.278882  ,  5.2921934 , ...,  2.8200073 ,
          2.8000321 ,  2.7813725 ],
        [ 5.7066803 ,  5.7069206 ,  5.6825843 , ...,  3.3369827 ,
          3.2048082 ,  3.130763  ]]], dtype=float32)
Coordinates:
  * longitude  (longitude) float32 148B 117.0 117.2 117.5 ... 125.5 125.8 126.0
  * latitude   (latitude) float32 148B 29.0 28.75 28.5 28.25 ... 20.5 20.25 20.0
  * time       (time) datetime64[ns] 3MB 1979-01-01 ... 2020-12-31T23:00:00

In [39]:
save_file = wind_anom_tw.to_dataset(name="sqrt_wind_speed_anomaly")
save_file.to_netcdf("/lfs/home/yanlan/climada/tc-risk/ERA5/wind_anom_tw.nc")